# **Лабораторная работа №9. "Построение моделей прогноза неуспевающих на курсе"**
---

FC-1.1-Б Способен теоретически обосновать и осуществить на практике оптимизацию
ML-моделей по техническим показателям эффективности (быстродействие,
объем используемой памяти), используя передовые способы обучения (AutoML,
нейродинамика, алгоритмы квантования и оптимизация на уровне данных)

## Цель работы:

изучить способы построения и автотюнинга пайплайнов, исследовать различные алгоритмы обучения, кросс-валидации и варианты генерации признаков для улучшения метрик.

## **Описание кейса**

Для организации учебного процесса стали использоваться в большом количестве онлайн-курсы. Но без контроля и соответствующего сопровождения обучение с применением онлайн-ресурсов может дать очень плачевный результат. Необходимо внедрять элементы контроля и сопровождения обучения студентов на онлайн-курсах.

Одной из таких задач построения интеллектуального тьютора онлайн-оубчения является задача прогноза конечного результата обучения на онлайн-курсе: сможет ли он успешно завершить курс в зависимости от его результатов работы в первой части курса.

Эту задачу можно решать разными способами.
1. Можно попробовать построить зависимость финального балла от результатов выполнения пройденных заданий курса. И в зависимости от предсказания определить - на какую оценку претендует пользователь и сможет ли он успешно завершить курс.
2. Можно решать задачу классификации, попытаться определить - к какому классу следует отнести данного слушателя, к классу успешно завершивших или к противоположному классу? Отнести его к классу тех, кто завершил курс на отлично или на удовлетворительно или не завершил.

В этом кейсе вы должны будете:
- исследовать различные алгоритмы и модели обучения классификатора учащихся по результатам выполнения первых пройденных заданий;
- оценить верхнюю границу точности классификатора на имеющихся данных;
- выбрать pipeline, применить автотюнинг и сравнить полученные результаты с лучшей моделью.

В качестве безлайна будем использовать логистическую регрессию.Вам нужно будет исследовать возможность улучшить данный результат за счет тюнинга пайплайна выбора признаков, выбора модели и параметров.

----------

В предыдуще лабораторной работе мы уже выполнили предподготовку данных:
- проанализировали, предобработали и разметили скачанные с курса данные - журнал оценок;
- отделили результаты первой половины курса и подготовили выборку для обучения и оценки моделей;
- исследовали полезность добавления дополнительных признаков (полиномиальных, логических).

После выбора метрик, построения и исследования моделей необходимо применить autoML и сравнить полученный результат с имеющимися моделями.

## **1. Загрузка и знакомство с исходными данными**
---
В качестве исходных данных мы будем использовать результаты выгрузки журнала оценок с онлайн-курса, который содержит результаты выполнения заданий онлайн-курса слушателями (объектно-признаковая матрица) и их финальные баллы (последний или 1-й столбец).

In [ ]:
# импорт необходимых библиотек работы с данными
import numpy as np
np.set_printoptions(precision=3)

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

### **Чтение данных из файла**

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# DIR PATH to DATA
dir_path = "/content/drive/My Drive/Colab Notebooks/teacher/INTO ML/cases/case01/data/" # указать каталог, из которого будет загружаться файл данных
os.chdir(dir_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# выбор файла данных
dir_path = "./"
course_file_name = "CompMath.csv"
csv_file_path = dir_path + course_file_name

# Чтение данных из файла *.csv
data = pd.read_csv(csv_file_path, sep=";")
print(data.shape)
data.tail()

(630, 43)


,Имя,Учреждение (организация),sum,Тест:1.1.5. Знакомство с Python (ТЕСТ) (Значение),Тест:1.2.9. Базовые типы данных в Python (ТЕСТ) (Значение),Тест:1.2.10. Управляющие структуры Python (ТЕСТ) (Значение),Тест:1.3.11. Функции и методы в Python (ТЕСТ) (Значение),"Тест:1.3.12. Кортежи,списки,словари в Python (ТЕСТ) (Значение)",Тест:1.4.11. Работа с массивами NumPy (ТЕСТ) (Значение),Тест:1.4.12. Работа с файлами и построение графиков (ТЕСТ) (Значение),...,Виртуальная лаборатория программирования:2.5.3. Программируем управление мобильным роботом (ВПЛ) (Значение),Пакет SCORM:2.3.4. Метод дихотомии и метод хорд (ВИДЕО) (Значение),Тест:ИТОГОВЫЙ ТЕСТ ПО РАЗДЕЛУ - РЕШЕНИЕ СИСТЕМ УРАВНЕНИЙ (Значение),Тест:3.1.5. Одномерная интерполяция (ТЕСТ) (Значение),Тест:3.2.5. Интерполяция поверхностей (ТЕСТ) (Значение),Тест:3.0.5. Постановки задач аппроксимации функций (ТЕСТ) (Значение),Тест:4.1.7. Линейный регрессионный анализ (ТЕСТ) (Значение),Виртуальная лаборатория программирования:4.1.8 Прогноз итоговой суммы баллов на онлайн-курсе (Значение),Тест:4.2.5. Логистическая регрессия (ТЕСТ) (Значение),"Виртуальная лаборатория программирования:4.2.6 Классификация слушателей онлайн-курса (Значение),,,,,,,,,,,"
625,Владислав,Волгатех,"50,04",2,5,5,5,"4,5","3,38","3,17",...,-,-,-,-,-,-,-,-,-,"-,,,,,,,,,"
626,Илья,Политехнический институт (филиал) СВФУ в г. Ми...,0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,"-,,,,,,,,,,,,,"
627,МГУ_Саранск_104,ФГБОУ ВО «Национальный исследовательский Мордо...,"85,04","2,58",5,5,5,5,4,"2,17",...,-,-,12,-,-,-,-,-,-,"-,,,,,,,,"
628,Антон,ПГТУ,"35,08","1,13",4,5,5,5,"3,25",-,...,-,-,-,-,-,-,-,-,-,"-,,,,,,,,,"
629,Анастасия,ННГУ Лобачевского,"64,89","2,23","3,5",4,4,5,4,"2,67",...,-,-,-,-,-,-,-,-,-,"-,,,,,,,,"


### **Описание датасета**
---
Данный датасет получен в результате прохождения студентами онлайн-курса "Программирование в Python и методы вычислений" на портале портала открытого образования Волгатеха (https://mooped.net/).
Это результат экспорта журнала оценок и очистки от чувствительной информации.

Полное описание датасета см. в предыдущей лаб.работе.

## **1. Исследуем и оцениваем модели классификаторов**
---

### **Загрузка и формирование массивов входных - выходных данных**

In [ ]:
# **3.1 Загружаем данные из файла**
datafile = dir_path + course_file_name + "_part1_tiny.csv"
data_3 = pd.read_csv(datafile, sep=";")
data_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439 entries, 0 to 438
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   T:1.1.5.     439 non-null    float64
 1   T:1.2.9.     439 non-null    float64
 2   T:1.2.10.    439 non-null    float64
 3   VPL:1.2.11.  439 non-null    float64
 4   VPL:1.2.12.  439 non-null    float64
 5   VPL:1.2.13.  439 non-null    float64
 6   T:1.3.11.    439 non-null    float64
 7   T:1.3.12.    439 non-null    float64
 8   VPL:1.3.13.  439 non-null    float64
 9   VPL:1.3.14.  439 non-null    float64
 10  VPL:1.3.15.  439 non-null    float64
 11  VPL:1.3.16.  439 non-null    float64
 12  T:1.4.11.    439 non-null    float64
 13  T:1.4.12.    439 non-null    float64
 14  VPL:1.4.13.  439 non-null    float64
 15  VPL:1.4.14.  439 non-null    float64
 16  VPL:1.4.15.  439 non-null    float64
 17  sumball      439 non-null    float64
 18  label        439 non-null    int64  
dtypes: float

In [ ]:
# Выделяем входные и выходные параметры в отдельные массивы
X = data_3.values[:, 0:-2]
y = data_3['sumball']
labels = data_3['label']

X.shape, y.shape, labels.shape

((439, 17), (439,), (439,))

### **1.1. Строим логистическую регрессию**
---

Используем кросс-валидацию для построения линейной зависимости. Но прежде чем применить кросс-валидацию необходимо разделить выборку на обучающую и тестовую.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Get the indices of the full dataset
indices = np.arange(len(X))

# Split X, y, and their corresponding indices
x_train, x_test, labels_train, labels_test, train_indices, test_indices = train_test_split(
    X, labels, indices, test_size=0.25, random_state=42
)

print(f"Train indices shape: {train_indices.shape}")
print(f"Test indices shape: {test_indices.shape}")

Train indices shape: (329,)
Test indices shape: (110,)


### **3.3. Построение логистической регрессии без дополнительных предикторов**
--------

Т.е. будем строить простую модель вида:

$$y_m(t) = sigmoid(a_0 + a_1*x_1 + ...+ a_k*x_k)$$


In [ ]:
from sklearn.linear_model import LogisticRegression
# здесь Ваш код ...
lr = LogisticRegression()
lr_model = lr.fit(x_train, labels_train)

print('a0=', lr_model.intercept_)
print(lr_model.coef_)

a0= [-14.719]
[[-0.587  0.306 -0.795 -0.064  0.442  0.325  0.778  0.631 -0.004  0.241
   1.028  0.264  0.952  0.933  0.615  1.167  0.967]]


### **1.3. Проанализируем полученную модель**
---

Оценим ее точность на тестовой выборке

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# точность прогноза тех, кто успешно завершит обучение
labels_predicted = lr_model.predict(x_test)
print("accuracy = {:.4f}".format(accuracy_score(labels_test, labels_predicted)))
print('f1_score=', f1_score(labels_test, labels_predicted))

sum(labels_predicted), sum(labels_test)

accuracy = 0.9727
f1_score= 0.9629629629629629


(np.int64(42), 39)

# Задания для студентов
### **1. Пробуем улучшить модель**
---
Попробуйте улучшить полученную модель. Повысить accuracy или f1_score.

Чтобы улучшить модель можно попробовать:

- применить другой метод или другие методы; например, RandomForest
- использовать кросс-валидацию;
- добавить полиномиальные предикторы;
- добавить бинаризацию признаков;
- потюнить параметры модели

#### добавляем полиномиальные признаки

In [ ]:
# здесь ваш код

#### добавляем бинарные признаки

In [ ]:
# здесь ваш код

## 2. **autoML**
---
Примените автотюнинг параметров генерации и подготовки признаков и параметров модели. Для этого:

- создайте свой Pipeline, который включает генерацию предкиторов модели и обучение модели;
- используйте GridSearchCV для настройки параметров Pipeline.
- проанализируйте полученные модели


### **3. Сделайте выводы по проведенному исследованию**
